# identity recipe — validate the recipe path (Phase 1)
Phase-0 proved the flux_residual seam == PuLID bit-identically. This confirms the RECIPE wiring:
`runner.run("identity", {id_image, prompt})` -> `_run_identity` (loads the shipped PuLID encoder via the identity
bridge, injects through flux_residual). PASS if ArcFace-sim to the reference ≈ Phase-0 (~0.76) and clearly beats a
no-identity baseline. OPEN-WEIGHT (PuLID) — NOT training-free. Runtime: 80GB A100.

In [ ]:
import subprocess, os
for _ in range(3):
    if subprocess.call(["pip","install","-q","git+https://github.com/huggingface/diffusers.git"])==0: break
!pip install -q transformers accelerate sentencepiece protobuf hf_transfer scikit-image
!pip install -q insightface facexlib onnxruntime-gpu timm einops ftfy opencv-python-headless
subprocess.run(["rm","-rf","flux-recipes"])
subprocess.run(["git","clone","-q","-b","main","https://github.com/remyxai/flux-recipes.git"])

In [ ]:
import sys, torch, numpy as np, cv2
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"flux-recipes")
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
assert torch.cuda.is_available(); print("GPU:",torch.cuda.get_device_name(0))
from flux_modular import RecipeRunner
runner=RecipeRunner(steps=20)
from PIL import Image, ImageDraw
from skimage import data
from IPython.display import display
REF=Image.fromarray(data.astronaut()).convert("RGB").resize((1024,1024))
print("runner ready")

## Run `identity` through the recipe path + measure identity

In [ ]:
PROMPT="a portrait photo of a person, studio lighting"
idimg = runner.run("identity", {"id_image": REF, "prompt": PROMPT}, id_weight=1.0, seed=0)
base  = runner.run("identity", {"id_image": REF, "prompt": PROMPT}, id_weight=0.0, seed=0)   # id off -> no identity

# ArcFace-sim via the PuLID encoder the recipe just loaded (identity bridge caches it)
from flux_modular.identity import _PULID
enc=_PULID["enc"]
def arc_emb(pil):
    fi=enc.app.get(cv2.cvtColor(np.asarray(pil.convert("RGB")),cv2.COLOR_RGB2BGR))
    if not fi: return None
    fi=sorted(fi,key=lambda x:(x['bbox'][2]-x['bbox'][0])*(x['bbox'][3]-x['bbox'][1]))[-1]
    e=fi['embedding']; return e/(np.linalg.norm(e)+1e-9)
def face_sim(a,b):
    ea,eb=arc_emb(a),arc_emb(b)
    return float(np.dot(ea,eb)) if (ea is not None and eb is not None) else None
sid, sbase = face_sim(idimg,REF), face_sim(base,REF)
panels=[("reference",REF,None),("id_weight=0 (baseline)",base,f"sim={sbase}"),("identity (id_weight=1)",idimg,f"sim={sid:.3f}")]
c=320; g=Image.new("RGB",(len(panels)*c+(len(panels)+1)*6,c+34),"white"); d=ImageDraw.Draw(g)
for j,(n,im,s) in enumerate(panels):
    x=6+j*(c+6); g.paste(im.resize((c,c)),(x,4)); d.text((x+4,c+8),str(n)[:36],fill="black")
    if s: d.text((x+4,c+20),str(s)[:36],fill="black")
display(g)
print(f"\nArcFace-sim to reference: identity={sid:.3f}  baseline(id_weight=0)={sbase}")
print("PASS if identity ~= Phase-0 (~0.76) and clearly > baseline -> the `identity` recipe works end-to-end.")